# General Conference Analysis - Interactive Exploration

This notebook demonstrates:
1. Scraping and caching conference talks
2. Temporal analysis of word/phrase trends
3. Embeddings-based semantic analysis
4. Topic clustering and visualization

## Understanding Embeddings

**Embeddings** are vector representations of text that capture semantic meaning. Think of them as coordinates in a high-dimensional space where:
- Similar concepts are close together
- Different concepts are far apart
- Mathematical operations can reveal relationships

For example:
- "faith" and "belief" will be close in embedding space
- "faith" and "automobile" will be far apart
- You can search by meaning, not just keywords

In [ ]:
# Import libraries
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date

# Import our custom modules
from conference_analysis.scraper import ConferenceScraper
from conference_analysis.temporal_analysis import TemporalAnalyzer
from conference_analysis.embeddings import EmbeddingsAnalyzer

# Set up plotting
sns.set_style('whitegrid')
%matplotlib inline

print("✅ Libraries imported successfully!")

## Step 1: Scrape or Load Conference Talks

This will scrape all General Conference talks from 1971 to present.
**Note**: First run will take 30-60 minutes. Results are cached for future use.

In [ ]:
# Initialize scraper with cache
scraper = ConferenceScraper(cache_file='../data/raw/talks.csv')

# Update cache (will only fetch new talks if cache exists)
talks = scraper.update_cache(delay=1.0)

print(f"\nTotal talks: {len(talks)}")
print(f"Date range: {talks['date'].min()} to {talks['date'].max()}")
print(f"\nSample talks:")
talks[['date', 'speaker', 'title']].head(10)

## Step 2: Temporal Analysis - Word and Phrase Trends

Track how specific words and phrases change in frequency over time.

In [ ]:
# Initialize temporal analyzer
temporal = TemporalAnalyzer(talks)

# Track specific words over time
words_to_track = ['faith', 'hope', 'charity', 'love', 'service', 'obedience']

# Create interactive plot
fig = temporal.plot_word_trends(
    words_to_track,
    time_grouping='decade',
    normalize=True,
    interactive=True
)
fig.show()

In [ ]:
# Track phrases over time
phrases_to_track = [
    'plan of salvation',
    'first vision',
    'book of mormon',
    'temple work'
]

fig = temporal.plot_phrase_trends(
    phrases_to_track,
    time_grouping='decade',
    normalize=True,
    interactive=True
)
fig.show()

In [ ]:
# Compare different decades
unique_1970s, unique_2020s, common = temporal.compare_periods(1970, 2020, time_grouping='decade')

print("Top words unique to 1970s:")
print(unique_1970s[:15])
print("\nTop words unique to 2020s:")
print(unique_2020s[:15])
print("\nCommon words:")
print(common[:15])

In [ ]:
# Generate word cloud for a specific decade
wordcloud = temporal.wordcloud_by_period(2020, time_grouping='decade')

plt.figure(figsize=(15, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud for 2020s', fontsize=20)
plt.tight_layout()
plt.show()

## Step 3: Embeddings Analysis - Semantic Understanding

This section demonstrates the power of embeddings for deeper semantic analysis.

### What are we doing here?
1. Converting each talk into a vector (embedding) that captures its meaning
2. Using cosine similarity to find talks with similar meanings
3. Clustering talks by their semantic content
4. Visualizing the semantic space in 2D

In [ ]:
# Initialize embeddings analyzer
# Using 'all-MiniLM-L6-v2' - a fast, efficient model
# For better quality (but slower), try 'all-mpnet-base-v2'

embeddings = EmbeddingsAnalyzer(
    talks,
    model_name='all-MiniLM-L6-v2',
    cache_dir='../data/processed'
)

# Generate embeddings (cached, so only slow on first run)
print("Generating embeddings...")
print("This takes 5-10 minutes on first run, then it's cached.")
embeddings.generate_embeddings()
print("✅ Embeddings ready!")

### Semantic Search - Find by Meaning, Not Keywords

Traditional search matches keywords. Semantic search matches **meaning**.

Try searching for concepts, not just words!

In [ ]:
# Semantic search example
query = "overcoming trials and adversity"

results = embeddings.semantic_search(query, top_k=10)

print(f"Top talks related to: '{query}'\n")
for idx, row in results.iterrows():
    print(f"{row['similarity']:.3f} - {row['title']}")
    print(f"          by {row['speaker']} ({row['date'].year})")
    print()

In [ ]:
# Try your own searches!
# Examples:
# - "raising faithful children"
# - "finding peace in difficult times"
# - "the importance of forgiveness"
# - "service and charity"

my_query = "the power of prayer"
results = embeddings.semantic_search(my_query, top_k=5)
print(results[['speaker', 'title', 'similarity']])

### Find Similar Talks

Given one talk, find others that discuss similar themes.

In [ ]:
# Pick a random talk and find similar ones
random_talk_idx = np.random.randint(0, len(talks))
reference_talk = talks.iloc[random_talk_idx]

print(f"Reference talk:")
print(f"  {reference_talk['title']}")
print(f"  by {reference_talk['speaker']} ({reference_talk['date'].year})")
print(f"\nSimilar talks:\n")

similar = embeddings.find_similar_talks(random_talk_idx, top_k=5)
for idx, row in similar.iterrows():
    print(f"{row['similarity']:.3f} - {row['title']}")
    print(f"          by {row['speaker']} ({row['date'].year})")
    print()

### Topic Clustering

Automatically discover natural groupings of talks based on their content.
This reveals recurring themes across all of General Conference.

In [ ]:
# Cluster talks into topics
n_clusters = 15
clustered = embeddings.cluster_talks(n_clusters=n_clusters)

# Examine each cluster
for cluster_id in range(n_clusters):
    summary = embeddings.get_cluster_summary(clustered, cluster_id, top_n=5)
    
    print(f"\n{'='*60}")
    print(f"CLUSTER {cluster_id} ({summary['size']} talks)")
    print(f"{'='*60}")
    
    print("\nMost representative talks:")
    for talk in summary['representative_talks'][:5]:
        print(f"  - {talk['title']}")
        print(f"    by {talk['speaker']}")
    
    print("\nTop speakers in this cluster:")
    for speaker, count in list(summary['top_speakers'].items())[:3]:
        print(f"  - {speaker}: {count} talks")

### Visualize the Semantic Space

See how talks relate to each other in 2D space.
Talks that are close together discuss similar themes.

In [ ]:
# Visualize embeddings colored by decade
# Sample 1000 talks for faster visualization
fig = embeddings.visualize_embeddings_2d(
    color_by='decade',
    sample_size=1000,
    method='pca'
)
fig.show()

# You can interact with this plot:
# - Hover over points to see talk details
# - Zoom in on clusters
# - See how talks from different decades are distributed

### Temporal Semantic Analysis

Combine embeddings with temporal analysis to see how discussion of a concept evolves.

In [ ]:
# Track how talks about "faith" have evolved
time_periods = [
    (1970, 1979),
    (1980, 1989),
    (1990, 1999),
    (2000, 2009),
    (2010, 2019),
    (2020, 2024)
]

evolution = embeddings.temporal_semantic_shift(
    concept="faith and belief in God",
    time_periods=time_periods,
    top_k=5
)

# Show top talks per period
for period in evolution['period'].unique():
    print(f"\n{'='*60}")
    print(f"Period: {period}")
    print(f"{'='*60}")
    period_talks = evolution[evolution['period'] == period]
    for idx, row in period_talks.iterrows():
        print(f"  {row['similarity']:.3f} - {row['title']}")
        print(f"            by {row['speaker']}")

## Step 4: Custom Analysis

Use the cells below for your own exploration!

In [ ]:
# Your custom analysis here


## Next Steps

Ideas for further exploration:

1. **Speaker Analysis**: Track how individual speakers' themes evolve over their ministry
2. **Topic Modeling**: Use LDA or BERTopic for more sophisticated topic discovery
3. **Sentiment Analysis**: Analyze emotional tone across different time periods
4. **Network Analysis**: Build a network of related talks and speakers
5. **Predictive Analysis**: Train models to predict which talks will be most impactful
6. **Web Dashboard**: Build an interactive Streamlit or Plotly Dash app

## Learning Resources

To learn more about embeddings and NLP:

- [Sentence Transformers Documentation](https://www.sbert.net/)
- [Understanding Word Embeddings](https://jalammar.github.io/illustrated-word2vec/)
- [Hugging Face NLP Course](https://huggingface.co/learn/nlp-course/chapter1/1)
- [Fast.ai NLP Course](https://www.fast.ai/)